# 多因子选股 + 树模型 完整流程演示

本 notebook 演示从数据获取到归因分析的完整流程。

**流程概览：**
1. 数据获取与清洗
2. 因子计算
3. 标签构建
4. 因子预处理（去极值、标准化、中性化）
5. 时间序列滚动划分
6. 模型训练与预测
7. 回测
8. 评估
9. 归因分析

## 0. 环境准备

In [ ]:
# 添加项目根目录到 Python 路径
import sys
from pathlib import Path

# 项目根目录（notebook 在 notebooks/ 下，src/ 在上一级）
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"项目根目录: {PROJECT_ROOT}")

In [ ]:
# 导入公共工具
from utils import load_config, setup_logger, get_data_path, get_output_path

# 加载配置
config = load_config(PROJECT_ROOT / "config" / "config.yaml")
logger = setup_logger("demo", config.get("log_level", "INFO"))

print("配置加载完成！")
print(f"  股票池: {config['data']['universe']}")
print(f"  时间范围: {config['data']['start_date']} ~ {config['data']['end_date']}")
print(f"  标签周期: {config['labels']['forward_days']} 日")

---
## 1. 数据获取与清洗

从 akshare 获取原始数据，保存为本地 parquet 文件。
首次运行会下载数据（耗时较长），后续运行直接读取本地文件。

In [ ]:
from data_loader import (
    get_index_constituents,
    download_daily_prices,
    download_valuation_indicators,
    download_index_data,
    clean_data
)

# 获取成分股列表
constituents = get_index_constituents(config, logger)
stock_codes = constituents["code"].tolist()
print(f"\n成分股数量: {len(stock_codes)}")
print(f"前 10 只: {stock_codes[:10]}")

In [ ]:
# 下载日频行情（首次运行耗时较长，约 30-60 分钟）
daily_prices = download_daily_prices(config, stock_codes, logger)
print(f"\n行情数据形状: {daily_prices.shape}")

In [ ]:
# 下载估值指标
valuation = download_valuation_indicators(config, stock_codes, logger)
print(f"\n估值数据形状: {valuation.shape}")

In [ ]:
# 下载指数行情
index_data = download_index_data(config, logger)
print(f"\n指数数据形状: {index_data.shape}")

In [ ]:
# 数据清洗
cleaned_data = clean_data(config, logger)
print(f"\n清洗后市场数据形状: {cleaned_data['market_data'].shape}")

---
## 2. 因子计算

计算六大类因子：动量、反转、波动、量价、估值、质量

In [ ]:
from features import compute_all_factors

factors = compute_all_factors(config, logger)
print(f"\n因子数据形状: {factors.shape}")
print(f"因子列表: {factors.columns.tolist()}")

---
## 3. 标签构建

构建未来 20 日收益率的横截面标准化标签

In [ ]:
from labels import build_labels

labels = build_labels(config, logger)
print(f"\n标签数据形状: {labels.shape}")
print(f"标签统计:\n{labels['label'].describe()}")

---
## 4. 因子预处理

去极值 → 标准化 → 中性化

In [ ]:
from preprocess import preprocess_factors

factors_processed = preprocess_factors(config, logger)
print(f"\n预处理后因子形状: {factors_processed.shape}")

---
## 5. 时间序列滚动划分

生成多个训练/验证窗口，模拟真实滚动预测

In [ ]:
from split import run_split

windows_data = run_split(config, logger)
print(f"\n共 {len(windows_data)} 个滚动窗口")

# 查看第一个窗口
if windows_data:
    w = windows_data[0]
    print(f"窗口 0: 训练 {len(w['X_train'])} 样本, 验证 {len(w['X_valid'])} 样本")

---
## 6. 模型训练与预测

训练四种模型：线性回归、随机森林、XGBoost、LightGBM

In [ ]:
from models import train_all_models

predictions = train_all_models(config, logger)
print(f"\n训练完成，共 {len(predictions)} 个模型的预测结果")

---
## 7. 回测

将预测值转化为投资组合，模拟真实交易

In [ ]:
from backtest import run_backtest

backtest_results = run_backtest(config, logger)
print(f"\n回测完成，共 {len(backtest_results)} 个模型的结果")

---
## 8. 评估

计算 Rank IC、ICIR、夏普、最大回撤等指标

In [ ]:
from evaluation import run_evaluation

evaluation_summary = run_evaluation(config, logger)
print("\n评估汇总:")
print(evaluation_summary)

---
## 9. 归因分析

特征重要性、SHAP 值、因子相关性

In [ ]:
from analysis import run_analysis

run_analysis(config, logger)
print("\n归因分析完成！图表保存在 outputs/figures/ 目录")

---
## 10. 结果汇总

所有结果保存在 `outputs/results/` 目录，图表在 `outputs/figures/` 目录。

In [ ]:
import os

results_dir = PROJECT_ROOT / "outputs" / "results"
figures_dir = PROJECT_ROOT / "outputs" / "figures"

print("结果文件:")
for f in sorted(results_dir.glob("*")):
    print(f"  {f.name}")

print("\n图表文件:")
for f in sorted(figures_dir.glob("*")):
    print(f"  {f.name}")